###**We validated the fraud engine using rule-consistency queries, threshold checks, and referential validation between Gold tables to ensure business correctness.**

## **VALIDATE SCORING LOGIC**

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE fraud_score !=
(
    CASE WHEN duplicate_claim_status = 'TRIGGERED' THEN 40 ELSE 0 END +
    CASE WHEN payment_integrity_status = 'TRIGGERED' THEN 30 ELSE 0 END +
    CASE WHEN claim_value_status = 'TRIGGERED' THEN 20 ELSE 0 END +
    CASE WHEN billing_pattern_status = 'TRIGGERED' THEN 20 ELSE 0 END +
    CASE WHEN provider_risk_level = 'MEDIUM' THEN 15
         WHEN provider_risk_level = 'HIGH' THEN 30
         ELSE 0 END
);

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE
    (fraud_score >= 60 AND fraud_flag != 'YES')
 OR (fraud_score BETWEEN 30 AND 59 AND fraud_flag != 'REVIEW')
 OR (fraud_score < 30 AND fraud_flag != 'NO');


-- score >= 60 → YES
-- 30–59 → REVIEW
-- <30 → NO

## **## VALIDATE TRIGGER SOURCE LOGIC**

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold. gold_fraud_signals
WHERE fraud_trigger_source = 'BOTH'
AND NOT (
    provider_risk_level IN ('MEDIUM','HIGH')
    AND (
        duplicate_claim_status='TRIGGERED'
        OR claim_value_status='TRIGGERED'
        OR payment_integrity_status='TRIGGERED'
        OR billing_pattern_status='TRIGGERED'
    )
);


--Claims marked BOTH must actually have both signals

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE fraud_trigger_source = 'CLAIM_ANOMALY'
AND provider_risk_level IN ('MEDIUM','HIGH');

## **VALIDATE BUSINESS RULES THEMSELVES**

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE claim_value_status='TRIGGERED'
AND claim_amount < 75000;

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE payment_integrity_status='TRIGGERED'
AND paid_amount <= claim_amount * 1.10;

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE billing_pattern_status='TRIGGERED'
AND claim_amount <= provider_avg_claim * 2;

In [0]:
%sql
SELECT claim_id, COUNT(*)
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE duplicate_claim_status='TRIGGERED'
GROUP BY claim_id
HAVING COUNT(*) = 1;

## **VALIDATE REFERENTIAL CONSISTENCY BETWEEN GOLD TABLES**

In [0]:
%sql
SELECT s.provider_id, s.total_claims, COUNT(g.claim_id) suspicious_claims
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_claim_summary s
LEFT JOIN dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals g
  ON s.provider_id = g.provider_id
GROUP BY s.provider_id, s.total_claims
HAVING COUNT(g.claim_id) > s.total_claims;

## **DATA SANITY CHECKS (Business sense checks)**

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE claim_amount < 0 OR paid_amount < 0;

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE paid_amount > claim_amount
AND payment_integrity_status != 'TRIGGERED';

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE claim_amount = 0
AND fraud_flag = 'YES';

In [0]:
%sql
SELECT *
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE paid_amount > claim_amount
AND payment_integrity_status != 'TRIGGERED';